In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp,current_timestamp

In [0]:
dbutils.widgets.text(
    "bronze_catalog",
     "dbr_dev")
dbutils.widgets.text(
    "bronze_schema",
    "artemzharkov10_bronze"
)

dbutils.widgets.text(
    "silver_catalog",
     "dbr_dev")
dbutils.widgets.text(
    "silver_schema",
    "artemzharkov10_silver"
)

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
    

In [0]:
df_bronze = spark.read.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.gdelt_history_bronze")
# print(f"Bronze row count: {df_bronze.count()}")

In [0]:
df_clean = df_bronze.dropna(subset=["SqlDate","GlobalEventID","SourceUrl","Actor1Name"])
df_remove_dublicate = df_clean.dropDuplicates(["GlobalEventID"])

df_silver = df_remove_dublicate.withColumn("Silver_Convertation_Date", current_timestamp())

(df_silver.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.gdelt_history_silver"))

In [0]:
# b = df_bronze.count()
# s = df_silver.count()
# print(f"Bronze records: {b}")
# print(f"Silver records: {s}")
# print(f"Records is filtered (removed): {b-s}")